<a href="https://colab.research.google.com/github/meghanavanamala/predictiveanalysis/blob/recommand_books_to-users/recommand_books_tousers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# --------------------------
# 1. Sample data
# --------------------------

# User-Book reading matrix (1 = read/liked, 0 = not read)
reading_data = pd.DataFrame({
    'Book1': [1, 0, 1, 1],
    'Book2': [0, 1, 1, 0],
    'Book3': [1, 1, 1, 1],
    'Book4': [0, 1, 0, 0],
    'Book5': [1, 0, 0, 1],
}, index=['UserA', 'UserB', 'UserC', 'UserD'])

# Book info: genre and reading mode
book_info = {
    'Book1': {'genre': 'fiction', 'mode': 'ebook'},
    'Book2': {'genre': 'romance', 'mode': 'audiobook'},
    'Book3': {'genre': 'fiction', 'mode': 'print'},
    'Book4': {'genre': 'self-help', 'mode': 'ebook'},
    'Book5': {'genre': 'fiction', 'mode': 'audiobook'}
}

# User preferences
user_preferences = {
    'UserA': {'genre': 'fiction', 'mode': 'ebook'},
    'UserB': {'genre': 'romance', 'mode': 'audiobook'},
    'UserC': {'genre': 'fiction', 'mode': 'print'},
    'UserD': {'genre': 'self-help', 'mode': 'ebook'}
}

# --------------------------
# 2. Recommendation function
# --------------------------

def recommend_books(user_id, top_n=3):
    # Step 1: Calculate user similarity
    similarity = cosine_similarity(reading_data)
    sim_df = pd.DataFrame(similarity, index=reading_data.index, columns=reading_data.index)

    # Step 2: Find similar users (exclude self)
    similar_users = sim_df[user_id].drop(user_id).sort_values(ascending=False)

    # Step 3: Get weighted scores of books from similar users
    book_scores = pd.Series(0, index=reading_data.columns, dtype='float64')
    for other_user, score in similar_users.items():
        book_scores += reading_data.loc[other_user] * score

    # Step 4: Remove already read books
    already_read = reading_data.loc[user_id][reading_data.loc[user_id] > 0].index
    book_scores = book_scores.drop(already_read, errors='ignore')

    # Step 5: Try filters (strict → loose → none)
    user_pref = user_preferences[user_id]
    genre_pref = user_pref['genre']
    mode_pref = user_pref['mode']

    def filter_books(condition_fn):
        return {
            book: score for book, score in book_scores.items()
            if condition_fn(book_info[book])
        }

    # Try strict match: genre + mode
    filtered = filter_books(lambda info: info['genre'] == genre_pref and info['mode'] == mode_pref)
    if not filtered:
        # Try genre only
        filtered = filter_books(lambda info: info['genre'] == genre_pref)
    if not filtered:
        # Try mode only
        filtered = filter_books(lambda info: info['mode'] == mode_pref)
    if not filtered:
        # Final fallback: return all unread books scored by similarity
        filtered = book_scores.to_dict()

    # Step 6: Sort and return top N
    recommended = pd.Series(filtered).sort_values(ascending=False).head(top_n)
    return recommended

# --------------------------
# 3. Run recommender
# --------------------------

user = 'UserA'
results = recommend_books(user)

print(f"\n📚 Book Recommendations for {user}:")
if results.empty:
    print("No matching books found.")
else:
    for book, score in results.items():
        info = book_info[book]
        print(f" - {book} (Genre: {info['genre']}, Mode: {info['mode']}) - Score: {round(score, 2)}")



📚 Book Recommendations for UserA:
 - Book4 (Genre: self-help, Mode: ebook) - Score: 0.33
